## Flight Operations project langGraph

Roxana D. and Tiffany N.

In [ ]:
%pip install -U langgraph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# choose llm model
# llm =

In [ ]:
from dataclasses import dataclass
from datetime import datetime


# disruption state

class Disruption(TypeDict):  # info from initial report
  flight_number: str
  depart_time: datetime
  date: datetime  # not sure if needed if it's included in depart_time
  # arrive_time: datetime
  start_location: str
  destination: str
  # could also have all of this info from one string like below
  # disruption_event: str
  status_confirmed: bool
  no_change_msg: str

class CentralAgent:
  def __init__(self, llm, flight_status_tool):
    self.llm = llm

    self.llm_tools = self.llm.bind_tools([flight_status_tool])

  def __call__(self, state: Disruption)
  # we could change this to the disruption event in general without the numbers and times separate if too tedious
  # like event = state.get("disruption_event")

  disrupt_flight_num = state.get("flight_number") # technically, would the flight checker only need the flight number to check updates??
  disrupt_depart_time = state.get("depart_time")
  disrupt_date = state.get("date")
  disrupt_start_loc = state.get("start_location")
  disrupt_dest = state.get("destination")

  # central agent verifies with flight status api tool
  prompt = ChatPromptTemplate.from_messages([
      ("system", "Use the flight status tool to only verify if the flights had details change. Let me know it any flight details have chnaged."),
      ("user", "{disrupt_flight_num}")
  ])

# should we add an output parser?
  chain = prompt | self.llm_tools
  response = chain.invoke({"disrupt_flight_num": disrupt_flight_num})

  # get results from tool call

  delayed = False

  if delayed == True: # would get from api tool, this is just a placeholder for now
    return {"status_confirmed": True} # would give update that flight is changed and lead to gatemanagement and passenger connections

  else:
    msg = "Your flight is still on time and has not been changed. However, I will keep checking every 10 minutes to give you potential updates."
    return {"status_confirmed": False, "no_change_msg": msg}
    # leads to confirmation about no status update

In [ ]:
central_agent = CentralAgent(llm=llm, flight_status_tool=flight_status_tool)

# initializing graph w disruption and central agent
workflow = StateGraph(Disruption)

workflow.add_node("central_agent", central_agent)

def no_status_update(state: Disruption):
  print({state['no_change_msg']})
  return state

#node for unchanged flight stat.
workflow.add_node("no_status_update", no_status_update)

# if flight stat is changed, route to gate mgmt and connections
def status_changed(state: Disruption):
  if state.get("status_confirmed"):
    return["Gate_management", "Passenger_Connections"]

  else:
    return "no_status_update"

workflow.add_edge(START, "central_agent")
workflow.add_conditional_edges("central_agent", status_changed)

# no status update has to lead to the confirmation?
# workflow.add_edge("no_status_update", confirmation)